# 臺股市場溫度計｜Google Colab 執行器

這份 Notebook 可在不同電腦以 Colab **Run all** 重現最新市場溫度計，並透過 Google Colab 代理網址開啟完整 Streamlit 介面。預設使用 `preview`：只計算、顯示並備份結果，不寄信、不修改 Google Sheet。

第一次使用前，請在 Colab 左側「鑰匙」Secrets 建立 `FINLAB_API_TOKEN`，並允許此 Notebook 存取。若改用 `cloud_daily` 正式補跑，還需建立 `GMAIL_SENDER`、`GMAIL_APP_PASSWORD`、`EMAIL_RECIPIENTS`、`GOOGLE_SHEET_ID`、`GOOGLE_SERVICE_ACCOUNT_JSON`。

> 正式排程仍以 GitHub Actions 為主；Colab 是研究、驗證、Streamlit 臨時介面與人工補跑工具。代理網址只在本次 Colab runtime 存活期間有效。

In [ ]:
# 1. 執行設定
REPO_URL = "https://github.com/hh4832/taiwan-market-thermometer.git"
BRANCH = "main"
RUN_MODE = "preview"  # preview 或 cloud_daily
RUN_TESTS = True
LAUNCH_STREAMLIT = True  # Run all 完成後建立可點擊的 Colab 代理網址

assert RUN_MODE in {"preview", "cloud_daily"}


In [ ]:
# 2. 掛載 Google Drive、取得最新版程式碼並安裝套件
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path
import shutil
import subprocess

# 重跑此cell時，不能在即將刪除的repository內工作，否則git會因getcwd失敗而回傳128。
os.chdir("/content")
subprocess.run(["pkill", "-f", "streamlit run dashboard/app.py"], check=False)
repo_dir = Path("/content/taiwan-market-thermometer")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)],
    cwd="/content",
    check=True,
)
os.chdir(repo_dir)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "local-requirements.txt"], check=True)
commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Repository: {repo_dir}")
print(f"Git commit: {commit_hash}")


In [ ]:
# 3. 從 Colab Secrets 載入憑證（不寫入檔案、不 commit）
from google.colab import userdata

required_secrets = ["FINLAB_API_TOKEN"]
if RUN_MODE == "cloud_daily":
    required_secrets += [
        "GMAIL_SENDER",
        "GMAIL_APP_PASSWORD",
        "EMAIL_RECIPIENTS",
        "GOOGLE_SHEET_ID",
        "GOOGLE_SERVICE_ACCOUNT_JSON",
    ]

missing = []
for name in required_secrets:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        missing.append(name)
    else:
        os.environ[name] = value

if missing:
    raise RuntimeError(f"請先在 Colab Secrets 建立並開放：{', '.join(missing)}")
print(f"已載入 {len(required_secrets)} 個必要 Secrets；內容不會顯示。")


In [ ]:
# 4. Baseline validation
if RUN_TESTS:
    subprocess.run(["python", "-m", "unittest", "discover", "-s", "tests-python", "-v"], check=True)
else:
    print("已略過測試。")


In [ ]:
# 5. 建立不可覆蓋、可追溯的 Drive output archive
from datetime import datetime
from zoneinfo import ZoneInfo

taipei_now = datetime.now(ZoneInfo("Asia/Taipei"))
run_id = f"{taipei_now:%Y%m%d_%H%M%S}_{commit_hash[:12]}"
output_dir = Path("/content/drive/MyDrive/Quant_Research/taiwan-market-thermometer") / run_id
output_dir.mkdir(parents=True, exist_ok=False)
print(f"Output archive: {output_dir}")


In [ ]:
# 6. 執行市場溫度計
import json
import finlab
import pandas as pd

finlab.login(os.environ["FINLAB_API_TOKEN"])

if RUN_MODE == "preview":
    from dashboard.cloud_daily import build_snapshot
    from dashboard.data_service import load_live_0050_close, load_live_breadth, load_live_futures

    breadth = load_live_breadth()
    futures = load_live_futures()
    close = load_live_0050_close()
    snapshot = build_snapshot(breadth, futures, close, taipei_now)

    (output_dir / "snapshot.json").write_text(
        json.dumps(snapshot, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    breadth.tail(30).to_csv(output_dir / "breadth_latest_30.csv", encoding="utf-8-sig")
    futures.tail(30).to_csv(output_dir / "futures_latest_30.csv", encoding="utf-8-sig")
    close.tail(30).to_csv(output_dir / "0050_close_latest_30.csv", encoding="utf-8-sig")
    display(pd.DataFrame([snapshot]).T.rename(columns={0: "value"}))
else:
    from dashboard.cloud_daily import run

    exit_code = run()
    if exit_code != 0:
        raise RuntimeError(f"cloud_daily 執行失敗，exit_code={exit_code}")
    snapshot = {"mode": "cloud_daily", "status": "success"}
    (output_dir / "cloud_daily_result.json").write_text(
        json.dumps(snapshot, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Google Sheet 更新與 Email 寄送完成。")


In [ ]:
# 7. 寫入 run_info，確認結果可追溯且未覆蓋舊紀錄
run_info = [
    f"run_at_taipei={taipei_now.isoformat()}",
    f"run_mode={RUN_MODE}",
    f"repo_url={REPO_URL}",
    f"branch={BRANCH}",
    f"git_commit={commit_hash}",
    f"tests_run={RUN_TESTS}",
    f"output_dir={output_dir}",
]
(output_dir / "run_info.txt").write_text("\n".join(run_info) + "\n", encoding="utf-8")

print("執行完成：")
print("\n".join(run_info))
print("\n輸出檔案：")
for path in sorted(output_dir.iterdir()):
    print("-", path.name)


In [ ]:
# 8. 透過 Google Colab 代理網址開啟 Streamlit
if LAUNCH_STREAMLIT:
    import time
    import urllib.request
    from IPython.display import HTML, display
    from google.colab.output import eval_js

    subprocess.run(["pkill", "-f", "streamlit run dashboard/app.py"], check=False)
    streamlit_log = open("/content/taiwan_market_thermometer_streamlit.log", "w", encoding="utf-8")
    streamlit_process = subprocess.Popen(
        [
            "python", "-m", "streamlit", "run", "dashboard/app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
            # Colab是反向代理；關閉下列保護與壓縮，避免前端永久停在skeleton。
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
            "--server.enableWebsocketCompression", "false",
            "--server.fileWatcherType", "none",
            "--browser.gatherUsageStats", "false",
        ],
        cwd=repo_dir,
        stdout=streamlit_log,
        stderr=subprocess.STDOUT,
    )

    health_url = "http://127.0.0.1:8501/_stcore/health"
    for _ in range(30):
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(1)
    else:
        streamlit_log.flush()
        streamlit_log.close()
        log_tail = Path("/content/taiwan_market_thermometer_streamlit.log").read_text(encoding="utf-8", errors="replace")[-4000:]
        print(log_tail)
        raise RuntimeError("Streamlit 啟動失敗；上方為Streamlit log末段")

    streamlit_url = eval_js("google.colab.kernel.proxyPort(8501)")
    display(HTML(
        f"<a href='{streamlit_url}' target='_blank' "
        "style='display:inline-block;padding:12px 18px;background:#137333;color:white;"
        "text-decoration:none;border-radius:8px;font-weight:700'>"
        "開啟臺股市場溫度計 Streamlit 介面</a>"
    ))
    print(f"Streamlit PID: {streamlit_process.pid}")
    print("若仍停在灰色骨架，請先重新整理該頁；診斷log位於 /content/taiwan_market_thermometer_streamlit.log")
    print("此網址只在目前 Colab runtime 運作期間有效。")
else:
    print("LAUNCH_STREAMLIT=False，已略過 Streamlit 啟動。")


## 修改程式後的 Git 工作流

若你在 Colab 內修改了 `src` 或 `dashboard` 程式，請先重新執行測試，再依序檢查：

`git status` → `git diff` → commit → push。

不要 commit Colab Secrets、FinLab 原始資料或 Drive outputs。Private repository 推送時請使用 Colab Secret `GITHUB_TOKEN`，不要把 token 寫進 Notebook。